# __Preproccessing Data__

_I chose to complete the Data Preparation part of teh project here in python using the Pandas library_

In [23]:
# Import necessary libraries
import pandas as pd
from pathlib import Path

In [ ]:
# Load the dataset
churn_path = "telecom_customer_churn.csv"
dict_path = "telecom_data_dictionary.csv"
zipcode_path = "telecom_zipcode_population.csv"

churn = pd.read_csv(churn_path)
data_dict = pd.read_csv(dict_path, encoding="windows-1252", engine="python")
zipcode = pd.read_csv(zipcode_path)

# Display basic information about the dataset
churn.info()
churn.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 38 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   object 
 1   Gender                             7043 non-null   object 
 2   Age                                7043 non-null   int64  
 3   Married                            7043 non-null   object 
 4   Number of Dependents               7043 non-null   int64  
 5   City                               7043 non-null   object 
 6   Zip Code                           7043 non-null   int64  
 7   Latitude                           7043 non-null   float64
 8   Longitude                          7043 non-null   float64
 9   Number of Referrals                7043 non-null   int64  
 10  Tenure in Months                   7043 non-null   int64  
 11  Offer                              3166 non-null   objec

,Customer ID,Gender,Age,Married,Number of Dependents,City,Zip Code,Latitude,Longitude,Number of Referrals,...,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Customer Status,Churn Category,Churn Reason
0,0002-ORFBO,Female,37,Yes,0,Frazier Park,93225,34.827662,-118.999073,2,...,Credit Card,65.6,593.30,0.00,0,381.51,974.81,Stayed,NaN,NaN
1,0003-MKNFE,Male,46,No,0,Glendale,91206,34.162515,-118.203869,0,...,Credit Card,-4.0,542.40,38.33,10,96.21,610.28,Stayed,NaN,NaN
2,0004-TLHLJ,Male,50,No,0,Costa Mesa,92627,33.645672,-117.922613,0,...,Bank Withdrawal,73.9,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices
3,0011-IGKFF,Male,78,Yes,0,Martinez,94553,38.014457,-122.115432,1,...,Bank Withdrawal,98.0,1237.85,0.00,0,361.66,1599.51,Churned,Dissatisfaction,Product dissatisfaction
4,0013-EXCHZ,Female,75,Yes,0,Camarillo,93010,34.227846,-119.079903,3,...,Credit Card,83.9,267.40,0.00,0,22.14,289.54,Churned,Dissatisfaction,Network reliability


_Tidy column names_

In [25]:
def tidy_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.str.strip()          # remove leading/trailing spaces
                 .str.lower()
                 .str.replace(" ", "_")
                 .str.replace(r"[()]", "", regex=True)
    )
    return df

churn = tidy_cols(churn)
zipcode = tidy_cols(zipcode)

_Data‑type conversions_

In [26]:
# Identify numeric columns that should be numeric
numeric_like = [
    "age", "number_of_dependents", "zip_code", "latitude", "longitude",
    "number_of_referrals", "tenure_in_months", "avg_monthly_long_distance_charges",
    "avg_monthly_gb_download", "monthly_charge", "total_charges",
    "total_refunds", "total_extra_data_charges", "total_long_distance_charges",
    "total_revenue"
]

for col in numeric_like:
    if col in churn.columns:
        churn[col] = pd.to_numeric(churn[col], errors="coerce")

# Convert booleans encoded as Yes/No
bool_like = [
    "married", "phone_service", "multiple_lines", "online_security",
    "online_backup", "device_protection_plan", "premium_tech_support",
    "streaming_tv", "streaming_movies", "streaming_music",
    "unlimited_data", "paperless_billing"
]
for col in bool_like:
    if col in churn.columns:
        churn[col] = churn[col].map({"Yes": True, "No": False})

_Handle missing values_

In [27]:
# Numeric: fill with median
for col in numeric_like:
    if col in churn.columns:
        median_val = churn[col].median()
        churn[col] = churn[col].fillna(median_val)

# Categorical: fill with 'Unknown'
categorical_cols = churn.select_dtypes(include="object").columns
churn[categorical_cols] = churn[categorical_cols].fillna("Unknown")

_Calculated fields_

In [29]:
# Average charge per month
churn["avg_charge_per_month"] = (
    churn["total_charges"] / churn["tenure_in_months"].replace({0: pd.NA})
)

# Tenure buckets (0‑12, 13‑24, etc.)
bins = list(range(0, churn["tenure_in_months"].max() + 12, 12))
labels = [f"{b}-{b+11}" for b in bins[:-1]]
churn["tenure_bucket"] = pd.cut(churn["tenure_in_months"], bins=bins, labels=labels, right=True)


_Combine with zipcode population_

In [30]:
combined = churn.merge(zipcode, how="left", on="zip_code", suffixes=("", "_zip"))
combined.rename(columns={"population": "zip_population"}, inplace=True)

_Persist cleaned data for Tableau_

In [ ]:
clean_path = "clean_telco_churn.csv"
combined_path = "clean_telco_churn_pop.csv"
churn.to_csv(clean_path, index=False)
combined.to_csv(combined_path, index=False)

In [32]:
churn.info()
churn.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 40 columns):
 #   Column                             Non-Null Count  Dtype   
---  ------                             --------------  -----   
 0   customer_id                        7043 non-null   object  
 1   gender                             7043 non-null   object  
 2   age                                7043 non-null   int64   
 3   married                            7043 non-null   bool    
 4   number_of_dependents               7043 non-null   int64   
 5   city                               7043 non-null   object  
 6   zip_code                           7043 non-null   int64   
 7   latitude                           7043 non-null   float64 
 8   longitude                          7043 non-null   float64 
 9   number_of_referrals                7043 non-null   int64   
 10  tenure_in_months                   7043 non-null   int64   
 11  offer                              7043 non

,customer_id,gender,age,married,number_of_dependents,city,zip_code,latitude,longitude,number_of_referrals,...,total_charges,total_refunds,total_extra_data_charges,total_long_distance_charges,total_revenue,customer_status,churn_category,churn_reason,avg_charge_per_month,tenure_bucket
0,0002-ORFBO,Female,37,True,0,Frazier Park,93225,34.827662,-118.999073,2,...,593.30,0.00,0,381.51,974.81,Stayed,Unknown,Unknown,65.922222,0-11
1,0003-MKNFE,Male,46,False,0,Glendale,91206,34.162515,-118.203869,0,...,542.40,38.33,10,96.21,610.28,Stayed,Unknown,Unknown,60.266667,0-11
2,0004-TLHLJ,Male,50,False,0,Costa Mesa,92627,33.645672,-117.922613,0,...,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices,70.212500,0-11
3,0011-IGKFF,Male,78,True,0,Martinez,94553,38.014457,-122.115432,1,...,1237.85,0.00,0,361.66,1599.51,Churned,Dissatisfaction,Product dissatisfaction,95.219231,12-23
4,0013-EXCHZ,Female,75,True,0,Camarillo,93010,34.227846,-119.079903,3,...,267.40,0.00,0,22.14,289.54,Churned,Dissatisfaction,Network reliability,89.133333,0-11
